In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from functools import partial

import numpy as np
from tqdm.notebook import tqdm

from src.data.dataloader import IterDataloader
from src.data.dataset import WikiTextDataset
from src.model.loss import CrossEntropyLoss, NegativeSamplingLoss
from src.model.model import CBOW
from src.model.optimizer import SGD
from src.utils.collate import wikitext_collate_fn

In [2]:
data = WikiTextDataset(
    "/home/tommeh/Projects/jetbrains-word2vec/data/wikitext-103/wiki.test.tokens",
    window_size=2,
    min_count=5,
    use_dynamic_window=True,
)

In [3]:
collate_fn = partial(wikitext_collate_fn, window_size=data.window_size)
dataloader = IterDataloader(data, 32, collate_fn)

In [ ]:
import csv
import os

num_epochs = 10
model = CBOW(vocab_size=data.tokenizer.vocab_size, n_dim=100)
neg_loss = NegativeSamplingLoss(word_counts=data.word_counts, k=10)
optimizer = SGD(lr=0.025, total_steps=len(data.corpus) * 0.2)

log_path = "../outputs/training_log.csv"
os.makedirs(os.path.dirname(log_path), exist_ok=True)
with open(log_path, "w", newline="") as log_file:
    writer = csv.writer(log_file)
    writer.writerow(["epoch", "avg_loss", "num_batches"])
    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        epoch_loss = 0.0
        n_batches = 0
        for batch, targets in tqdm(dataloader, desc="Batches", leave=False):
            logits = model.forward(batch)
            loss = neg_loss(logits, targets)
            delta_H = neg_loss.backward()
            model.backward(delta_H)
            optimizer.step(
                model.params() + neg_loss.params(), model.grads() + neg_loss.grads()
            )
            epoch_loss += loss
            n_batches += 1
        avg_loss = epoch_loss / n_batches
        writer.writerow([epoch, f"{avg_loss:.6f}", n_batches])
        log_file.flush()
        print(f"Epoch {epoch}: avg_loss={avg_loss:.4f} ({n_batches} batches)")
print(f"Training log saved to {log_path}")

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Batches: 0it [00:00, ?it/s]

Epoch 0: avg_loss=7.5070 (1187 batches)


Batches: 0it [00:00, ?it/s]

Epoch 1: avg_loss=6.2905 (1180 batches)


...

Epoch 7: avg_loss=3.2968 (1186 batches)


Batches: 0it [00:00, ?it/s]

Epoch 8: avg_loss=3.2700 (1176 batches)


Batches: 0it [00:00, ?it/s]

Epoch 9: avg_loss=3.2524 (1189 batches)
Training log saved to ../outputs/training_log.csv
